## Importing modules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display

### Importing from files

In [2]:
import sys
import os

# Get the absolute path to the `main/relativistic_dof/` directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../relativistic_dof/")))

# Now you can import your class or function
from relativistic_dof import RelativisticDOFRegistry

In [3]:
# --- Class deals with: evolution of the entropy relativistc degrees of freedom
RelativisticDOF = RelativisticDOFRegistry.get_method("fit")

## Thermall axion

We investigate the Figure 1 from 2205.01637, where we can notice few points: ($m_\mathrm{a}$, $T_{\mathrm{dec}}$). To calculate the extra relativistc degrees of freedom we used (2.14) and fractional axion abundance we used (2.15).

In [4]:
# -------------------- SET FORMULAS FROM PUBLICATION -------------------- #
def calculate_dNeff(T_dec):
    """
    Approximation: Instantaneous axion decoupling.
    
    Parameters:
    T_dec : float or array-like
        Decouple temperature in [eV].
        
    Returns:
    float or np.ndarray
        Extra relativistic degrees of freedom (ΔNeff).
    """
    # axion decouple
    x_dec = 1  # decouple from the plasma

    # Check if T_dec is array-like (list, tuple, or NumPy array)
    if isinstance(T_dec, (list, tuple, np.ndarray)):
        T_dec = np.array(T_dec)  # Convert to NumPy array for vectorized operations
        axion_decouple_dof = np.array([RelativisticDOF.compute_decoupling_dof(t, x_dec) for t in T_dec])  # My old approache
        # axion_decouple_dof = np.array([g_starS(t/x_dec) for t in T_dec])  # More accurate one
    else:
        axion_decouple_dof = np.array(RelativisticDOF.compute_decoupling_dof(T_dec, x_dec))  # My old approache
        # axion_decouple_dof = g_starS(T_dec/x_dec)

    # Formula (2.14) from your reference
    dNeff = 0.027 * (axion_decouple_dof / 106.75) ** (-4 / 3)

    return dNeff
    
def calculate_wa(T_dec, m_a):
    """
    Approximation: instantenous axion decouple
    T_dec: decouple temperature in [eV]
    m_a: axion mass in [eV]
    """
    # find extra relativistic degrees of freedom
    dNeff = calculate_dNeff(T_dec)

    # formula (2.15)
    wa = 0.011 * m_a * dNeff**(3/4)
    return wa

In [5]:
# -------------------- TAKE AXION MASS AND DECOUPLE TEMPERATURE -------------------- #
# Define the file path
file_path = "Figure1_points"  # Change to the actual file name

# Initialize storage structures
data_dict = {}
data_dict["m_a"] = np.array([])    # [eV]
data_dict["T_dec"] = np.array([])  # [GeV]
data_dict["dNeff"] = np.array([])
data_dict["w_a"] = np.array([])

# Read the file
with open(file_path, "r") as file:
    for line in file:
        # Skip comments or headers
        if line.startswith("#"):
            continue

        # Split the line into two values (m_a, T_d)
        values = line.strip().split(",")

        # Convert to floats and store
        m_a, T_d = float(values[0]), float(values[1])*10**9  # in [eV], [eV]

        # calculate other values
        dNeff = calculate_dNeff(T_d)
        w_a = calculate_wa(T_d, m_a)
        
        # appending
        data_dict["m_a"] = np.append(data_dict["m_a"], m_a)
        data_dict["T_dec"] = np.append(data_dict["T_dec"], T_d)
        data_dict["dNeff"] = np.append(data_dict["dNeff"], dNeff)
        data_dict["w_a"] = np.append(data_dict["w_a"], w_a)

In [6]:
# Create a DataFrame
df = pd.DataFrame({
    "Axion Mass (m_a) [eV]": data_dict["m_a"],
    "Decouple Temp (T_dec) [GeV]": data_dict["T_dec"]*10**(-9),
    "ΔNeff (dNeff)": data_dict["dNeff"],
    "Axion Abundance (w_a)": data_dict["w_a"]
})

display(df)

,Axion Mass (m_a) [eV],Decouple Temp (T_dec) [GeV],ΔNeff (dNeff),Axion Abundance (w_a)
0,0.000100,0.051642,0.388976,5.443958e-07
1,0.001001,0.056162,0.378082,5.307971e-06
2,0.010064,0.050765,0.391299,5.477058e-05
3,0.100721,0.067105,0.356963,5.116582e-04
4,1.003198,0.157676,0.164976,2.856574e-03
5,2.989257,0.279136,0.079109,4.904845e-03
6,9.992022,1.135063,0.047231,1.113561e-02
7,30.060043,0.365204,0.067630,4.385195e-02
8,100.480022,0.801828,0.051183,1.189366e-01


In [7]:
# -------------------- CROSS CHECK ------------------------------------------------------------------- #
# We take values from the axes of Figure 1. We compare those one which ones we calculate
# The only error here witch can be take into account is the differences in calculation
# relativistc degrees of freedom (our caluclation and guys from the publication)
T_dec_arr = np.array([1e-5, 1e-3, 1e-1, 1e1, 1e3])*10**(9)  # [eV]
dNeff_computed = calculate_dNeff(T_dec_arr)
dNeff_reference = [2.202, 0.590, 0.305, 0.039, 0.028]
relative_diff = abs(dNeff_reference - dNeff_computed)/dNeff_reference * 100

# Create a DataFrame
dCheck = pd.DataFrame({
    "Decouple Temp (T_dec) [GeV]": T_dec_arr*10**(-9),
    "ΔNeff computed": dNeff_computed,
    "ΔNeff reference": dNeff_reference,
    "relative error [%]": relative_diff
})

display(dCheck)

,Decouple Temp (T_dec) [GeV],ΔNeff computed,ΔNeff reference,relative error [%]
0,0.00001,2.203858,2.202,0.084365
1,0.00100,0.591036,0.590,0.175552
2,0.10000,0.304916,0.305,0.027578
3,10.00000,0.039380,0.039,0.974747
4,1000.00000,0.027905,0.028,0.339900


In [8]:
def calculate_Tncdm(kB_T_GeV):
    T_nu = (4/11)**(1/3)  # temperature of neutrinos [T_gamma]
    beta = 7/4 * 0.027

    # entropy DOF
    kB_T_eV = kB_T_GeV * 10**(9)  # change units from [GeV] to [eV]
    x_dec = 1  # instantenous decoupling
    g_starS = np.float64(RelativisticDOF.compute_decoupling_dof(kB_T_eV, x_dec))

    return T_nu * beta**(1/4) * (106.75/g_starS)**(1/3)
    

In [9]:
kb_T_GeV = 1*10**(-3) # [GeV]
Tncdm = calculate_Tncdm(kb_T_GeV)

print(f"kb_T_GeV: {kb_T_GeV} [GeV] coincides with Tncdm: {Tncdm} [T_gamma]")

kb_T_GeV: 0.001 [GeV] coincides with Tncdm: 0.7198113992566417 [T_gamma]


In [10]:
kb_T_GeV = 1.135063	 # [GeV]
Tncdm = calculate_Tncdm(kb_T_GeV)

print(f"m_a = 10 eV: kb_T_GeV: {kb_T_GeV} [GeV] coincides with Tncdm: {Tncdm} [T_gamma]")

m_a = 10 eV: kb_T_GeV: 1.135063 [GeV] coincides with Tncdm: 0.3827112524860587 [T_gamma]


In [11]:
kb_T_GeV = 0.050765		 # [GeV]
Tncdm = calculate_Tncdm(kb_T_GeV)

print(f"m_a = 0.01 eV: kb_T_GeV: {kb_T_GeV} [GeV] coincides with Tncdm: {Tncdm} [T_gamma]")

m_a = 0.01 eV: kb_T_GeV: 0.050765 [GeV] coincides with Tncdm: 0.6492958776666322 [T_gamma]
